In [1]:
import tensorflow as tf

batch_size = 32
img_height = 224
img_width = 224

# Diviser les données en 80% train et 20% pour validation/test
train_val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size = int(0.5 * len(val_test_ds))  # 50% de val_test_ds pour validation
test_size = len(val_test_ds) - val_size  # Reste pour test

val_ds = val_test_ds.take(val_size)  # Premier 50% pour validation
test_ds = val_test_ds.skip(val_size)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds)}")
print(f"Nombre de batches dans val_ds: {len(val_ds)}")
print(f"Nombre de batches dans test_ds: {len(test_ds)}")

Found 1968 files belonging to 3 classes.
Using 1575 files for training.
Found 1968 files belonging to 3 classes.
Using 393 files for validation.
Nombre de batches dans train_ds: 50
Nombre de batches dans val_ds: 6
Nombre de batches dans test_ds: 7


In [2]:
## chargement dataset maiis
# Diviser les données en 80% train et 20% pour validation/test
train_val_ds_maiis = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset maiis',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds_maiis = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset maiis',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size = int(0.5 * len(val_test_ds_maiis))  # 50% de val_test_ds pour validation
test_size = len(val_test_ds_maiis) - val_size  # Reste pour test

val_ds_maiis = val_test_ds_maiis.take(val_size)  # Premier 50% pour validation
test_ds_maiis = val_test_ds_maiis.skip(val_size)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds_maiis)}")
print(f"Nombre de batches dans val_ds: {len(val_ds_maiis)}")
print(f"Nombre de batches dans test_ds: {len(test_ds_maiis)}")

Found 1930 files belonging to 3 classes.
Using 1544 files for training.
Found 1930 files belonging to 3 classes.
Using 386 files for validation.
Nombre de batches dans train_ds: 49
Nombre de batches dans val_ds: 6
Nombre de batches dans test_ds: 7


In [3]:
## chargement dataset maiis
# Diviser les données en 80% train et 20% pour validation/test
train_val_ds_mixte = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset mixte',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds_mixte = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset mixte',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size_mixte = int(0.5 * len(val_test_ds_mixte))  # 50% de val_test_ds pour validation
test_size_mixte = len(val_test_ds_mixte) - val_size_mixte  # Reste pour test

val_ds_mixte = val_test_ds_maiis.take(val_size_mixte)  # Premier 50% pour validation
test_ds_mixte = val_test_ds_maiis.skip(val_size_mixte)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds_mixte)}")
print(f"Nombre de batches dans val_ds: {len(val_ds_mixte)}")
print(f"Nombre de batches dans test_ds: {len(test_ds_mixte)}")

Found 1940 files belonging to 3 classes.
Using 1552 files for training.
Found 1940 files belonging to 3 classes.
Using 388 files for validation.
Nombre de batches dans train_ds: 49
Nombre de batches dans val_ds: 6
Nombre de batches dans test_ds: 7


In [4]:
from tensorflow.keras.applications import ResNet50  # Remplacer ResNet101 par ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import time

# Définir les dimensions de l'image
img_height, img_width = 224, 224

# Charger le modèle ResNet50 pré-entraîné
base_model = ResNet50(input_shape=(img_height, img_width, 3),
                      include_top=False,
                      weights='imagenet')
base_model.trainable = False  # Geler les poids du modèle pré-entraîné

for layer in base_model.layers[-50:]:
    layer.trainable = True

# Ajouter une couche Global Average Pooling
global_average_layer = layers.GlobalAveragePooling2D()(base_model.output)

# Ajouter des couches fully connected supplémentaires avec Dropout
dense_1 = layers.Dense(1024, activation='relu')(global_average_layer)
dropout_1 = layers.Dropout(0.5)(dense_1)

dense_2 = layers.Dense(512, activation='relu')(dropout_1)
dropout_2 = layers.Dropout(0.5)(dense_2)

dense_3 = layers.Dense(256, activation='relu')(dropout_2)
dropout_3 = layers.Dropout(0.5)(dense_3)

dense_4 = layers.Dense(128, activation='relu')(dropout_3)
dropout_4 = layers.Dropout(0.5)(dense_4)

# Créer les sorties pour chaque nutriment (13 au total)
outputs = []
for nutrient in range(13):
    output = layers.Dense(3, activation='softmax', name=f'nutrient_{nutrient}')(dropout_4)
    outputs.append(output)

# Créer le modèle final avec ResNet50 en entrée et les 13 sorties en sortie
model = models.Model(inputs=base_model.input, outputs=outputs)

# Compilation du modèle avec des métriques adaptées pour chaque sortie
metrics = ['accuracy', Precision(name='precision'), Recall(name='recall')]
metrics_list = [metrics] * 13  # Appliquer les métriques à chaque nutriment

model.compile(optimizer='adam',
              loss=['categorical_crossentropy'] * 13,
              metrics=metrics_list)

# Afficher un résumé du modèle pour vérifier les couches et les sorties
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_pad (ZeroPadding2D)     │ (None, 230, 230, 3)       │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 112, 112, 64)      │           9,472 │ conv1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_bn (BatchNormalization) │ (None, 112, 112, 64)      │             256 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_relu (Activation)       │ (None, 112, 112, 64)      │               0 │ conv1_bn[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pad (ZeroPadding2D)     │ (None, 114, 114, 64)      │               0 │ conv1_relu[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pool (MaxPooling2D)     │ (None, 56, 56, 64)        │               0 │ pool1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 56, 56, 64)        │           4,160 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 56, 56, 64)        │          36,928 │ conv2_block1_1_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_2_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_2_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_3_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ conv2_block1_2_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 26,379,943 (100.63 MB)

 Trainable params: 19,742,759 (75.31 MB)

 Non-trainable params: 6,637,184 (25.32 MB)

In [5]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Définir le checkpoint pour sauvegarder les meilleurs poids
checkpoint = ModelCheckpoint('ResNet_50_weights.keras',
                             monitor='val_accuracy',
                             verbose=1,
                             mode='max',
                             save_best_only=True)

# Early stopping pour arrêter l'entraînement si la validation stagne
early = EarlyStopping(monitor="val_loss",
                      mode="min",
                      restore_best_weights=True,
                      patience=5)

# Liste des callbacks
callbacks_list = [checkpoint, early]

In [6]:
import time
# Entraînement du modèle tout en mesurant le temps
start_time = time.time()

history = model.fit(
    train_val_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=callbacks_list,
    verbose=True,
    shuffle=True
)

end_time = time.time()

# Afficher le temps d'entraînement
training_time = end_time - start_time
print(f"Temps d'apprentissage : {training_time} secondes")

# Calcul manuel du F1-score après l'entraînement
precision = history.history['precision'][-1]
recall = history.history['recall'][-1]
f1 = 2 * (precision * recall) / (precision + recall + tf.keras.backend.epsilon()) 
print(f'F1 Score: {f1:.4f}')

Epoch 1/15


C:\Users\Bamba\AppData\Roaming\Python\Python312\site-packages\keras\src\optimizers\base_optimizer.py:678: UserWarning: Gradients do not exist for variables ['kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias', 'kernel', 'bias'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - loss: 1.6950 - nutrient_0_accuracy: 0.3680 - nutrient_0_precision: 0.3671 - nutrient_0_recall: 0.2285

C:\Users\Bamba\AppData\Roaming\Python\Python312\site-packages\keras\src\callbacks\model_checkpoint.py:206: UserWarning: Can save best model only with val_accuracy available, skipping.
  self._save_model(epoch=epoch, batch=None, logs=logs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 442s 7s/step - loss: 1.6913 - nutrient_0_accuracy: 0.3688 - nutrient_0_precision: 0.3680 - nutrient_0_recall: 0.2278 - val_loss: 538.3503 - val_nutrient_0_accuracy: 0.3281 - val_nutrient_0_precision: 0.3281 - val_nutrient_0_recall: 0.3281
Epoch 2/15
50/50 ━━━━━━━━━━━━━━━━━━━━ 303s 6s/step - loss: 1.1425 - nutrient_0_accuracy: 0.5546 - nutrient_0_precision: 0.6842 - nutrient_0_recall: 0.2856 - val_loss: 3.8710 - val_nutrient_0_accuracy: 0.6042 - val_nutrient_0_precision: 0.6118 - val_nutrient_0_recall: 0.5417
Epoch 3/15
50/50 ━━━━━━━━━━━━━━━━━━━━ 303s 6s/step - loss: 0.6508 - nutrient_0_accuracy: 0.6468 - nutrient_0_precision: 0.7092 - nutrient_0_recall: 0.5054 - val_loss: 1.5376 - val_nutrient_0_accuracy: 0.6042 - val_nutrient_0_precision: 0.6242 - val_nutrient_0_recall: 0.5365
Epoch 4/15
50/50 ━━━━━━━━━━━━━━━━━━━━ 304s 6s/step - loss: 0.6415 - nutrient_0_accuracy: 0.6672 - nutrient_0_precision: 0.6867 - nutrient_0_recall: 0.5807 - val_loss: 0.5584 - val_nutr

KeyError: 'precision'

In [7]:
model.save("model/ResNet50_06_11.h5")

In [8]:
# Évaluation du modèle sur l'ensemble de test
results = model.evaluate(test_ds)

# Affichez les résultats
print(f"Résultats de l'évaluation : {results}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 19s 3s/step - loss: 0.4702 - nutrient_0_accuracy: 0.7023 - nutrient_0_precision: 0.7025 - nutrient_0_recall: 0.6865
Résultats de l'évaluation : [0.47745081782341003, 0.7014925479888916, 0.7020202279090881, 0.6915422677993774]


In [9]:

# Évaluation sur les données de validation
val_results = model.evaluate(test_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

7/7 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/step - loss: 0.4771 - nutrient_0_accuracy: 0.7175 - nutrient_0_precision: 0.7241 - nutrient_0_recall: 0.7130
Nombre total de résultats: 4
Résultats de l'évaluation: [0.48301708698272705, 0.7263681888580322, 0.7346938848495483, 0.7164179086685181]
La structure des résultats est différente de celle attendue.


In [10]:
# Évaluation du modèle sur l'ensemble de test
results = model.evaluate(test_ds_mixte)

# Affichez les résultats
print(f"Résultats de l'évaluation : {results}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 22s 3s/step - loss: 0.8451 - nutrient_0_accuracy: 0.6751 - nutrient_0_precision: 0.6799 - nutrient_0_recall: 0.6674
Résultats de l'évaluation : [0.8023673892021179, 0.6701030731201172, 0.6736842393875122, 0.6597937941551208]


In [11]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds_mixte)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

7/7 ━━━━━━━━━━━━━━━━━━━━ 20s 3s/step - loss: 0.7337 - nutrient_0_accuracy: 0.6664 - nutrient_0_precision: 0.6800 - nutrient_0_recall: 0.6507
Nombre total de résultats: 4
Résultats de l'évaluation: [0.7246427536010742, 0.6649484634399414, 0.6737967729568481, 0.6494845151901245]
La structure des résultats est différente de celle attendue.


In [12]:
# Évaluation du modèle sur l'ensemble de test
results = model.evaluate(test_ds_maiis)

# Affichez les résultats
print(f"Résultats de l'évaluation : {results}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 20s 3s/step - loss: 0.6208 - nutrient_0_accuracy: 0.6750 - nutrient_0_precision: 0.6740 - nutrient_0_recall: 0.6591
Résultats de l'évaluation : [0.6952484250068665, 0.6752577424049377, 0.678947389125824, 0.6649484634399414]


In [13]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds_maiis)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

7/7 ━━━━━━━━━━━━━━━━━━━━ 20s 3s/step - loss: 0.7397 - nutrient_0_accuracy: 0.6475 - nutrient_0_precision: 0.6535 - nutrient_0_recall: 0.6392
Nombre total de résultats: 4
Résultats de l'évaluation: [0.7527110576629639, 0.6494845151901245, 0.6542553305625916, 0.6340206265449524]
La structure des résultats est différente de celle attendue.


In [14]:
# Évaluation sur les données de validation
val_results = model.evaluate(val_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

6/6 ━━━━━━━━━━━━━━━━━━━━ 20s 3s/step - loss: 0.4555 - nutrient_0_accuracy: 0.6853 - nutrient_0_precision: 0.6887 - nutrient_0_recall: 0.6853
Nombre total de résultats: 4
Résultats de l'évaluation: [0.45631101727485657, 0.703125, 0.7105262875556946, 0.703125]
La structure des résultats est différente de celle attendue.


In [15]:
import time
from tensorflow.keras import backend as K

# Fonction pour calculer la moyenne d'une métrique
def mean_metric(metric_values):
    return sum(metric_values) / len(metric_values)

# Nombre de nutriments
num_nutrients = 13

# Fonction pour calculer les moyennes pour chaque nutriment
def mean_nutrient_metric(metric_name, history):
    metrics = []
    for nutrient in range(num_nutrients):
        key = f'nutrient_{nutrient}_{metric_name}'
        if key in history:
            metrics.append(history[key])
    return [mean_metric(metric) for metric in zip(*metrics)]  # Moyenne sur les époques

# Calcul des moyennes pour l'ensemble des époques (entraînement)
mean_train_loss = mean_metric(history.history['loss'])
mean_train_accuracy = mean_nutrient_metric('accuracy', history.history)
mean_train_precision = mean_nutrient_metric('precision', history.history)
mean_train_recall = mean_nutrient_metric('recall', history.history)

# Calcul du F1-score moyen pour l'ensemble des époques (entraînement)
f1_train_scores = [2 * (p * r) / (p + r + K.epsilon()) 
                   for p, r in zip(mean_train_precision, mean_train_recall)]
mean_f1_train = mean_metric(f1_train_scores)

# Calcul des moyennes pour l'ensemble des époques (validation)
mean_val_loss = mean_metric(history.history['val_loss'])
mean_val_accuracy = mean_nutrient_metric('accuracy', history.history)
mean_val_precision = mean_nutrient_metric('precision', history.history)
mean_val_recall = mean_nutrient_metric('recall', history.history)

# Calcul du F1-score moyen pour l'ensemble des époques (validation)
f1_val_scores = [2 * (p * r) / (p + r + K.epsilon()) 
                 for p, r in zip(mean_val_precision, mean_val_recall)]
mean_f1_val = mean_metric(f1_val_scores)

# Affichage des résultats
print(f"Moyenne de la perte sur l'ensemble d'entraînement : {mean_train_loss:.4f}")
print(f"Moyenne de l'accuracy sur l'ensemble d'entraînement : {mean_metric(mean_train_accuracy):.4f}")
print(f"Moyenne de la précision sur l'ensemble d'entraînement : {mean_metric(mean_train_precision):.4f}")
print(f"Moyenne du rappel sur l'ensemble d'entraînement : {mean_metric(mean_train_recall):.4f}")
print(f"Moyenne du F1-score sur l'ensemble d'entraînement : {mean_f1_train:.4f}")

print(f"Moyenne de la perte sur l'ensemble de validation : {mean_val_loss:.4f}")
print(f"Moyenne de l'accuracy sur l'ensemble de validation : {mean_metric(mean_val_accuracy):.4f}")
print(f"Moyenne de la précision sur l'ensemble de validation : {mean_metric(mean_val_precision):.4f}")
print(f"Moyenne du rappel sur l'ensemble de validation : {mean_metric(mean_val_recall):.4f}")
print(f"Moyenne du F1-score sur l'ensemble de validation : {mean_f1_val:.4f}")

Moyenne de la perte sur l'ensemble d'entraînement : 0.6668
Moyenne de l'accuracy sur l'ensemble d'entraînement : 0.6966
Moyenne de la précision sur l'ensemble d'entraînement : 0.7233
Moyenne du rappel sur l'ensemble d'entraînement : 0.6149
Moyenne du F1-score sur l'ensemble d'entraînement : 0.6568
Moyenne de la perte sur l'ensemble de validation : 340.0283
Moyenne de l'accuracy sur l'ensemble de validation : 0.6966
Moyenne de la précision sur l'ensemble de validation : 0.7233
Moyenne du rappel sur l'ensemble de validation : 0.6149
Moyenne du F1-score sur l'ensemble de validation : 0.6568
